# JungGo Podcast Studio — 3D GPU Render on Google Colab (Free T4)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/dpjungrana-code/junggo-podcast-3d/blob/studio-v1/render_on_colab.ipynb)

Run these 4 cells in order on a **Runtime -> Change runtime type -> T4 GPU** instance (or click **Runtime -> Run all**).

1. **Generate audio & timestamps locally** on your Mac (`python scripts/build_episode.py`) and `git push`.
2. **Run Cell 1–3 below** to clone/pull the latest episode from `https://github.com/dpjungrana-code/junggo-podcast-3d.git` and render the 1080p 3D MP4 directly to your Google Drive (`/content/drive/MyDrive/JungGoPodcast/episode_output.mp4`).

In [ ]:
# @title Cell 1: Environment & Dependencies (T4 GPU + Node.js 20 + EGL/Chromium)
!nvidia-smi

# Install Node.js 20+
!curl -fsSL https://deb.nodesource.com/setup_20.x | sudo -E bash - > /dev/null
!sudo apt-get update -qq
!sudo apt-get install -y -qq nodejs xvfb libegl1 libgles2 libnss3 libatk-bridge2.0-0 libgbm1 \
    libxkbcommon0 libxdamage1 libxcomposite1 libxrandr2 libpango-1.0-0 libcairo2 \
    fonts-noto-cjk && (sudo apt-get install -y -qq libasound2t64 || sudo apt-get install -y -qq libasound2)

!node -v && npm -v

# Install npm packages immediately if package.json is already in current directory
!if [ -f package.json ]; then (npm ci || npm install); else echo 'Ready for Cell 2 project sync.'; fi

In [ ]:
# @title Cell 2: Mount Google Drive & Sync Project
import os
from google.colab import drive

drive.mount('/content/drive')
OUTPUT_DIR = '/content/drive/MyDrive/JungGoPodcast'
os.makedirs(OUTPUT_DIR, exist_ok=True)
print('Google Drive output folder:', OUTPUT_DIR)

# Option A: Clone or pull the latest episode from GitHub
GITHUB_REPO_URL = 'https://github.com/dpjungrana-code/junggo-podcast-3d.git'  # @param {type:"string"}
PROJECT_DIR = '/content/junggo-podcast-3d'

if GITHUB_REPO_URL.strip():
    if os.path.exists(os.path.join(PROJECT_DIR, '.git')):
        %cd {PROJECT_DIR}
        !git pull --ff-only
    else:
        !git clone "{GITHUB_REPO_URL}" {PROJECT_DIR}
        %cd {PROJECT_DIR}
else:
    # Option B: Use project files uploaded into /content/junggo-podcast-3d or Drive zip
    DRIVE_ZIP = os.path.join(OUTPUT_DIR, 'junggo-podcast-3d.zip')
    if not os.path.exists(PROJECT_DIR) and os.path.exists(DRIVE_ZIP):
        !unzip -q "{DRIVE_ZIP}" -d /content/
    if os.path.exists(PROJECT_DIR):
        %cd {PROJECT_DIR}
    else:
        print('Upload your project to /content/junggo-podcast-3d or set GITHUB_REPO_URL above.')

# Install Node dependencies if needed & verify audio/episode JSON exist
!if [ -f package.json ] && [ ! -d node_modules ]; then (npm ci || npm install); fi
!ls -lh public/audio/current_episode.mp3 src/data/currentEpisode.json

In [ ]:
# @title Cell 3: Render 1080p 3D Video on T4 GPU -> Google Drive
import os
import subprocess

OUTPUT_MP4 = '/content/drive/MyDrive/JungGoPodcast/episode_output.mp4'
env = os.environ.copy()
env['REMOTION_FULL_QUALITY'] = '1'

attempts = [
    (
        'T4 GPU (angle-egl)',
        [
            'xvfb-run', '-a', 'npx', 'remotion', 'render',
            'JungGoPodcast3D', OUTPUT_MP4,
            '--gl=angle-egl', '--concurrency=4',
            '--chromium-flags={"ignore-gpu-blocklist":true,"enable-gpu":true,"disable-gpu-sandbox":true}',
        ],
    ),
    (
        'T4 GPU (egl)',
        [
            'xvfb-run', '-a', 'npx', 'remotion', 'render',
            'JungGoPodcast3D', OUTPUT_MP4,
            '--gl=egl', '--concurrency=4',
            '--chromium-flags={"ignore-gpu-blocklist":true,"enable-gpu":true,"disable-gpu-sandbox":true}',
        ],
    ),
    (
        'SwiftShader WebGL fallback',
        [
            'xvfb-run', '-a', 'npx', 'remotion', 'render',
            'JungGoPodcast3D', OUTPUT_MP4,
            '--gl=swangle', '--concurrency=2',
            '--chromium-flags={"use-gl":"angle","use-angle":"swiftshader-webgl","enable-unsafe-swiftshader":true}',
        ],
    ),
]

success = False
for label, cmd in attempts:
    print(f'Trying {label}:', ' '.join(cmd))
    res = subprocess.run(cmd, env=env)
    if res.returncode == 0:
        success = True
        break
    print(f'{label} exited with code {res.returncode}, trying next option...\n')

if not success:
    raise RuntimeError('All Remotion render attempts failed.')

size_mb = os.path.getsize(OUTPUT_MP4) / (1024 * 1024)
print(f'\nRender complete! Saved to {OUTPUT_MP4} ({size_mb:.2f} MB)')

In [ ]:
# @title Cell 4: Direct Download Backup (Optional)
from google.colab import files

files.download('/content/drive/MyDrive/JungGoPodcast/episode_output.mp4')